# Model Demo: PubMed RCT Sentence Classification

This notebook demonstrates how to load trained models and run predictions on sample sentences.

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import torch
import numpy as np
from transformers import AutoTokenizer

from data_loader import get_class_labels
from preprocess import preprocess_sentences
from tokenizer_utils import LSTMTTokenizer, TransformerTokenizer
from model_lstm import LSTMClassifier
from utils import set_seed

# Set seed for reproducibility
set_seed(42)

# Class labels
class_labels = get_class_labels()
id_to_label = {v: k for k, v in class_labels.items()}

## Sample Sentences

Define some sample sentences for testing the models.

In [ ]:
# Sample sentences for testing
sample_sentences = [
    "The study was conducted to evaluate the efficacy of aspirin in preventing cardiovascular events.",
    "Patients were randomly assigned to receive either aspirin or placebo.",
    "The primary outcome was the incidence of myocardial infarction.",
    "These findings suggest that aspirin may reduce the risk of heart disease.",
    "Further research is needed to confirm these results."
]

# Expected classes (for reference)
expected_classes = ["Objective", "Methods", "Methods", "Conclusions", "Conclusions"]

print("Sample sentences:")
for i, (sentence, expected) in enumerate(zip(sample_sentences, expected_classes)):
    print(f"{i+1}. {sentence}")
    print(f"   Expected: {expected}")
    print()

## Load LSTM Model

Load the trained LSTM model and make predictions.

In [ ]:
# Load LSTM model
def load_lstm_model(model_path='../results/lstm_model.pth', vocab_size=30000):
    model = LSTMClassifier(vocab_size=vocab_size)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.eval()
    return model

# Initialize tokenizer (you would need to load the trained vocab)
lstm_tokenizer = LSTMTTokenizer()

# For demo, we'll use a basic tokenizer without trained vocab
# In practice, you'd load the vocab from training
def predict_lstm(sentences, model, tokenizer):
    # Preprocess
    processed = preprocess_sentences(sentences)
    
    # Tokenize (using basic encoding)
    encoded = [tokenizer.encode(text, max_length=128) for text in processed]
    inputs = torch.tensor(encoded, dtype=torch.long)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).numpy()
    
    return preds

# Load model (uncomment when model exists)
# lstm_model = load_lstm_model()
# lstm_preds = predict_lstm(sample_sentences, lstm_model, lstm_tokenizer)

print("LSTM predictions would be computed here...")
# print("LSTM Predictions:")
# for sentence, pred in zip(sample_sentences, lstm_preds):
#     print(f"Text: {sentence}")
#     print(f"Predicted: {id_to_label[pred]}")
#     print()

## Load Transformer Model

Load the trained BERT-based model and make predictions.

In [ ]:
# Load transformer model
from transformers import AutoModelForSequenceClassification

def load_transformer_model(model_path='../results/transformer_model'):
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    return model, tokenizer

def predict_transformer(sentences, model, tokenizer):
    # Preprocess
    processed = preprocess_sentences(sentences)
    
    # Tokenize
    inputs = tokenizer(processed, truncation=True, padding=True, 
                      max_length=128, return_tensors='pt')
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1).numpy()
        probs = torch.softmax(outputs.logits, dim=1).numpy()
    
    return preds, probs

# Load model (uncomment when model exists)
# transformer_model, transformer_tokenizer = load_transformer_model()
# transformer_preds, transformer_probs = predict_transformer(sample_sentences, transformer_model, transformer_tokenizer)

print("Transformer predictions would be computed here...")
# print("Transformer Predictions:")
# for sentence, pred, prob in zip(sample_sentences, transformer_preds, transformer_probs):
#     confidence = prob[pred]
#     print(f"Text: {sentence}")
#     print(f"Predicted: {id_to_label[pred]} (confidence: {confidence:.3f})")
#     print()

## Comparison and Analysis

Compare predictions from both models and analyze results.

In [ ]:
# Comparison (uncomment when models are loaded)
print("Model Comparison:")
print("Sample | Expected | LSTM | Transformer")
print("-" * 50)

# for i, (sentence, expected) in enumerate(zip(sample_sentences, expected_classes)):
#     lstm_pred = id_to_label.get(lstm_preds[i], "N/A")
#     transformer_pred = id_to_label.get(transformer_preds[i], "N/A")
#     
#     print(f"{i+1} | {expected} | {lstm_pred} | {transformer_pred}")
#     print(f"   {sentence[:60]}...")
#     print()

print("Demo completed. Load actual trained models to see predictions.")

## Notes

- To run this demo, you need trained models saved in the `../results/` directory
- The LSTM model requires the vocabulary to be built during training
- The transformer model uses the standard BERT tokenizer
- Confidence scores are available for the transformer model

This notebook provides a template for model inference and comparison.